# 04 Statistical Analysis
This notebook applies statistical techniques, correlation analysis, and hypothesis testing to validate assumptions about margins, discounts, and shipping.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('data/processed/cleaned_dataset.csv')

## 1. Correlation Matrix
Understanding the linear relationships between numerical covariates.

In [ ]:
numeric_cols = ['sales', 'profit', 'discount', 'shipping_cost', 'order_quantity', 'product_base_margin']
correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Matrix for Numerical Features')
plt.show()

## 2. Hypothesis Testing: Discount impact on Profit
**H0**: Average profit is the same whether discount is < 5% or >= 5%.
**H1**: Average profit differs based on discount tiers.

In [ ]:
low_discount = df[df['discount'] < 0.05]['profit'].dropna()
high_discount = df[df['discount'] >= 0.05]['profit'].dropna()

t_stat, p_val = stats.ttest_ind(low_discount, high_discount, equal_var=False)
print(f'T-Statistic: {t_stat:.4f}')
print(f'P-Value: {p_val}')

if p_val < 0.05:
    print('Reject Null Hypothesis: Significant difference in profit based on discounts.')
else:
    print('Fail to reject Null Hypothesis')

## 3. Regression Analysis: Predicting Margin
Is unit price and order quantity significantly predicting base margins?

In [ ]:
df_valid = df.dropna(subset=['product_base_margin', 'unit_price', 'order_quantity', 'discount'])
X = df_valid[['unit_price', 'order_quantity', 'discount']]
Y = df_valid['product_base_margin']
X = sm.add_constant(X) # adding a constant

model = sm.OLS(Y, X).fit()
print(model.summary())